<a href="https://colab.research.google.com/github/ethane101/ghpcs-data/blob/main/ghpcs_vae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setting Up and Importing

In [ ]:
!pip install pytorch torchvision

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.2 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This er

In [ ]:
!pip install keras tensorflow

In [ ]:
from sklearn.datasets import make_classification
import sklearn as sk
import keras
import matplotlib.pyplot as plt
import numpy as np

import pandas as pd
import pickle
from google.colab import drive
drive.mount('/drive')

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


# Loading in the Data Sets and Concatenating Them

In [ ]:
with open('/drive/My Drive/GHP CS/research-project/data/processed-data2.pkl', 'rb') as cdata:
  surv_df = pickle.load(cdata)
  cnv_df = pickle.load(cdata)
  methy_df = pickle.load(cdata)
  mrna_df = pickle.load(cdata)
  mirna_df = pickle.load(cdata)

dfs = [surv_df, cnv_df, methy_df, mrna_df, mirna_df]
dfs_names = 'surv_df cnv_df methy_df mrna_df mirna_df'.split()
dfs[0].name = 'surv_df'
dfs[1].name = 'cnv_df'
dfs[2].name = 'methy_df'
dfs[3].name = 'mrna_df'
dfs[4].name = 'mirna_df'

## Concatenating the Data Sets

In [40]:
full_df = pd.concat(dfs[1:], axis=1) # full dataframe with cnv_df, methy_df, mrna_df, mirna_df (without surv_df)
full_df = full_df.to_numpy(dtype=np.float32)
full_df.shape

(177, 1049)

# Implementing the Variational Autoencoder
### Note: Must use T4 GPU for code to run!

In [41]:
sample_num = full_df.shape[0]
feature_num = full_df.shape[1]
info_num = 175 # 175 extracted features
redundant_num = feature_num - info_num

X = full_df
y = np.random.rand(177,)  # (177,) dimension, this data set (y) doesn't matter and only exists to use existing functions!
n_inputs = X.shape[1]
print(X.shape, y.shape)

(177, 1049) (177,)


In [42]:
# split into train test sets -> 30% testing, 70% for training
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.30, random_state=1)
print(X_train.shape)
print(X_test.shape)

(123, 1049)
(54, 1049)


In [43]:
# define encoder -> same shape as an individual record
visible = keras.Input(shape=(n_inputs,))
# encoder level 1 -> expands to twice the size
e = keras.layers.Dense(n_inputs*2)(visible)
e = keras.layers.BatchNormalization()(e)
e = keras.layers.LeakyReLU()(e)
# encoder level 2 -> goes back to original size
e = keras.layers.Dense(n_inputs)(e)
e = keras.layers.BatchNormalization()(e)
e = keras.layers.LeakyReLU()(e)
# bottleneck -> implements dimensionality reduction in the latent space
n_bottleneck = 150
bottleneck = keras.layers.Dense(n_bottleneck)(e)

In [44]:
# define decoder, level 1 -> takes in latent features
d = keras.layers.Dense(n_inputs)(bottleneck)
d = keras.layers.BatchNormalization()(d)
d = keras.layers.LeakyReLU()(d)
# decoder level 2 -> expands features to twice the original size
d = keras.layers.Dense(n_inputs*2)(d)
d = keras.layers.BatchNormalization()(d)
d = keras.layers.LeakyReLU()(d)
# output layer -> returns features back to original size, finishes decoding
output = keras.layers.Dense(n_inputs, activation='linear')(d)
# define autoencoder model
model = keras.Model(inputs=visible, outputs=output)

In [45]:
# compile autoencoder model -> adam used because it is computationally efficient and performs stochastic gradient descent, used in tandem with MeanSquaredError
model.compile(optimizer='adam', loss='mse')

In [60]:
# plot the autoencoder
# keras.utils.plot_model(model, 'autoencoder_no_compress.png', show_shapes=True)

In [62]:
def vaeTrain(epoch_num):
  # fit the autoencoder model to reconstruct input
  history = model.fit(X_train, X_train, epochs=epoch_num, verbose=2, validation_data=(X_test,X_test))
  # plot loss
  return (history.history['loss'][-1], history.history['val_loss'][-1])

In [65]:
def optimizeVAE():
  losses = {}
  for epoch_num in range(X_train.shape[0]):
    losses[epoch_num] = vaeTrain(epoch_num)

optimizeVAE()

KeyError: 'loss'

In [ ]:
# define an encoder model (without the decoder)
encoder = keras.Model(inputs=visible, outputs=bottleneck)
keras.utils.plot_model(encoder, 'encoder_no_compress.png', show_shapes=True)
# save the encoder to file
encoder.save('encoder.keras')